## 什么是中间件（Middleware）
中间件指在执行循环中，位于请求和响应之间的组件。它们可以处理请求、修改响应、执行额外的操作等。
在Langchain中使用中间件能够帮助我们构建更复杂的Agent应用程序。

### Managing Long Conversations with Middleware
当模型上下文越来越长时，可能会遇到很多问题，例如上下文超出模型窗口限制、推理成本增加、模型遗忘重点、响应速度变慢等。使用中间件可以帮助我们管理长对话，保持对话的相关性和效率。
以下是一些常见的中间件策略：
1. **摘要中间件**：在每次请求之前，对之前的对话进行摘要，保留关键信息，减少上下文长度。
2. **记忆中间件**：使用外部存储（如数据库）来保存对话历史，并在需要时检索相关信息，避免将所有对话历史都放入上下文中。
3. **过滤中间件**：根据特定规则过滤掉不相关的对话内容，只保留与当前请求相关的信息。
4. **分段中间件**：将长对话分成多个段落，每次只处理当前段落，保持上下文的相关性和效率。
通过使用这些中间件策略，我们可以更好地管理长对话，提高模型的响应质量和效率。

In [4]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
llm = ChatOpenAI(
    model='my_local_model',
    base_url='http://127.0.0.1:1234/v1',
    api_key='none_for_need',
)


In [6]:
agent = create_agent(llm, 
                     tools=[],
                     checkpointer= InMemorySaver(),
                     middleware=[
                         SummarizationMiddleware(
                             model = llm,
                             trigger=('tokens', 50), # 当上下文超过 50 token 时触发总结
                             keep=('messages', 1) # 总结完成后，只保留最近 1 条消息，其余历史变成 summary
                         )
                     ]
                     )

In [7]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint
messages = [
    HumanMessage(content='我最近在学习 Python 编程。'),
    AIMessage(content='Python 是一门非常适合初学者的语言。'),
    HumanMessage(content='我已经学会了列表、字典和函数。'),
    AIMessage(content='很好，这些是 Python 的核心基础。'),
    HumanMessage(content='我最近开始学习类和面向对象编程。'),
    AIMessage(content='面向对象是 Python 很重要的一部分。'),
    HumanMessage(content='我还想进一步学习异步编程和网络请求。'),
]
responses = agent.invoke({'messages':messages},
                         {'configurable':{'thread_id':'1'}})

pprint(responses)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nGuide and tutor the user in learning Python programming, specifically focusing on transitioning from core fundamentals to Object-Oriented Programming (OOP).\n\n## SUMMARY\n- User has successfully learned Python basics: lists, dictionaries, and functions.\n- User is currently beginning to study classes and Object-Oriented Programming (OOP).\n- AI acknowledged these topics as core foundations and highlighted the importance of OOP in Python.\n- The conversation is in the early informational stage; no specific learning strategies, complex decisions, or rejected options have been established yet.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\n- Provide structured explanations and practical code examples for core Python OOP concepts (classes, objects, attributes, methods, inheritance, encapsulation, polymorphism).\n- Assess the user's current understanding through targeted questions or small cod

In [8]:
pprint(responses['messages'][0].content)

('Here is a summary of the conversation to date:\n'
 '\n'
 '## SESSION INTENT\n'
 'Guide and tutor the user in learning Python programming, specifically '
 'focusing on transitioning from core fundamentals to Object-Oriented '
 'Programming (OOP).\n'
 '\n'
 '## SUMMARY\n'
 '- User has successfully learned Python basics: lists, dictionaries, and '
 'functions.\n'
 '- User is currently beginning to study classes and Object-Oriented '
 'Programming (OOP).\n'
 '- AI acknowledged these topics as core foundations and highlighted the '
 'importance of OOP in Python.\n'
 '- The conversation is in the early informational stage; no specific learning '
 'strategies, complex decisions, or rejected options have been established '
 'yet.\n'
 '\n'
 '## ARTIFACTS\n'
 'None\n'
 '\n'
 '## NEXT STEPS\n'
 '- Provide structured explanations and practical code examples for core '
 'Python OOP concepts (classes, objects, attributes, methods, inheritance, '
 'encapsulation, polymorphism).\n'
 "- Assess the us

## 自定义中间件
单次调用，在运行前后：  
@before_agent  
@after_agent

---

多次调用，在每次调用模型前后：  
@before_model  
@after_model
